# Entrega 3: Modelado de Datos para Visualización
En este notebook, realizaremos el preprocesamiento necesario para construir los modelos de datos que consumirá Tableau.
Evaluaremos tres enfoques de arquitectura de datos:
1. **Tabla Plana (Flat Table):** Un único archivo desnormalizado.
2. **Esquema Estrella (Star Schema):** Un diseño normalizado con una Tabla de Hechos central y Tablas de Dimensión, optimizado para Tableau.
3. **Copo de Nieve (Snowflake Schema):** Normalización adicional de las dimensiones.

In [1]:
import pandas as pd
import numpy as np
import os

# Configuración de visualización
pd.set_option('display.max_columns', None)

## 1. Carga de Datos y Generación de Clave Primaria
Cargamos el dataset limpio proveniente de la Entrega 2. Como los identificadores de hogar originales (CONGLOME, VIVIENDA, HOGAR) fueron eliminados en la reducción de dimensionalidad, generaremos una **Clave Primaria Subrogada (`ID_HOGAR`)** para poder relacionar las tablas.

In [2]:
# Cargar el dataset que limpiamos en la Entrega 2
df = pd.read_csv('../Data/limpio/Sumaria-2024_limpio.csv')

# CORRECCIÓN: Evitar que pandas borre los ceros a la izquierda del UBIGEO
df['UBIGEO'] = df['UBIGEO'].astype(str).str.zfill(6)

# Generar la clave primaria subrogada ID_HOGAR
df.insert(0, 'ID_HOGAR', range(1, len(df) + 1))

print(f"Dataset cargado con éxito. Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
display(df.head(3))

Dataset cargado con éxito. Filas: 33691, Columnas: 37


,ID_HOGAR,MES,UBIGEO,DOMINIO,ESTRATO,POBREZA,POBREZAV,INGHOG2D,GASHOG2D,MIEPERHO,GRU11HD,GRU21HD,GRU31HD,GRU41HD,GRU51HD,GRU61HD,GRU71HD,GRU81HD,FACTOR07,BRECHA_HOG,EN_DEFICIT,ING_PERCAPITA,GAS_PERCAPITA,BRECHA_PERCAPITA,MES_NUM,DEPARTAMENTO,LOG_INGHOG2D,LOG_GASHOG2D,LOG_BRECHA_PERCAP,GRU11HD_PCT,GRU21HD_PCT,GRU31HD_PCT,GRU41HD_PCT,GRU51HD_PCT,GRU61HD_PCT,GRU71HD_PCT,GRU81HD_PCT
0,1,1,010101,Sierra Norte,"De 20,000 a 49,999 habitantes",No Pobre,No Vulnerable,52162.609375,34188.218750,2,3845.363770,874.391602,1006.782715,1307.083496,3339.000000,3247.260742,1318.787598,3219.265869,79.816757,17974.390625,0,26081.304688,17094.109375,8987.195312,1,Amazonas,10.862140,10.439666,9.103667,0.1125,0.0256,0.0294,0.0382,0.0977,0.0950,0.0386,0.0942
1,2,1,010101,Sierra Norte,"De 20,000 a 49,999 habitantes",No Pobre,No Vulnerable,40832.042969,40164.945312,3,12792.489258,2384.777832,916.000000,665.926880,2408.242676,2768.000000,895.084717,752.953064,79.816757,667.097656,0,13610.680990,13388.315104,222.365885,1,Amazonas,10.617247,10.600775,5.408811,0.3185,0.0594,0.0228,0.0166,0.0600,0.0689,0.0223,0.0187
2,3,1,010101,Sierra Norte,"De 20,000 a 49,999 habitantes",No Pobre,No Vulnerable,15098.497070,12308.838867,1,2150.485107,603.583740,2589.000000,581.172546,2047.000000,233.000000,41.478962,557.595886,79.816757,2789.658203,0,15098.497070,12308.838867,2789.658203,1,Amazonas,9.622417,9.418154,7.934033,0.1747,0.0490,0.2103,0.0472,0.1663,0.0189,0.0034,0.0453


## 2. Modelo 1: Tabla Plana (Flat Table)
El Modelo de Tabla Plana es el formato actual del dataset. Exportaremos este formato para tener el "Modelo Base" contra el cual comparar.

In [3]:
# Crear directorio para los modelos si no existe
os.makedirs('../Data/modelo', exist_ok=True)
os.makedirs('../Data/modelo/tabla_plana', exist_ok=True)

# Exportar la Tabla Plana
df.to_csv('../Data/modelo/tabla_plana/Sumaria-2024_flat.csv', index=False, encoding='utf-8-sig')
print("Modelo de Tabla Plana exportado exitosamente.")

Modelo de Tabla Plana exportado exitosamente.


## 3. Modelo 2: Esquema Estrella (Star Schema)
Para optimizar el rendimiento en Tableau y reducir la redundancia, crearemos un esquema estrella. 
Separaremos la información categórica en **Tablas de Dimensión** y dejaremos las métricas continuas en la **Tabla de Hechos**.

### 3.1. Dimensión Geografía (`dim_geografia`)
Contiene los atributos de ubicación del hogar. La clave primaria será el `UBIGEO`.

In [4]:
# Seleccionar columnas para la dimensión geografía
dim_geografia = df[['UBIGEO', 'DEPARTAMENTO', 'DOMINIO', 'ESTRATO']].drop_duplicates().reset_index(drop=True)
dim_geografia.insert(0, 'ID_GEOGRAFIA', range(1, len(dim_geografia) + 1))

print(f"Dimensión Geografía: {dim_geografia.shape[0]} registros únicos.")
display(dim_geografia.head(3))

Dimensión Geografía: 1905 registros únicos.


,ID_GEOGRAFIA,UBIGEO,DEPARTAMENTO,DOMINIO,ESTRATO
0,1,010101,Amazonas,Sierra Norte,"De 20,000 a 49,999 habitantes"
1,2,010307,Amazonas,Selva,"De 2,000 a 19,999 habitantes"
2,3,010701,Amazonas,Selva,"De 20,000 a 49,999 habitantes"


### 3.2. Dimensión Tiempo (`dim_tiempo`)
Contiene los atributos temporales. La clave primaria será `MES_NUM`.

In [5]:
# Seleccionar columnas para la dimensión tiempo con nombres en español
nombres_meses = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
    7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}
dim_tiempo = df[['MES_NUM']].drop_duplicates().sort_values('MES_NUM').reset_index(drop=True)
dim_tiempo['MES'] = dim_tiempo['MES_NUM'].map(nombres_meses)

print(f"Dimensión Tiempo: {dim_tiempo.shape[0]} registros únicos.")
display(dim_tiempo.head(3))

Dimensión Tiempo: 12 registros únicos.


,MES_NUM,MES
0,1,Enero
1,2,Febrero
2,3,Marzo


### 3.3. Dimensión Pobreza (`dim_pobreza`)
Agrupa las clasificaciones de pobreza y vulnerabilidad, además del indicador de déficit. Generaremos una clave subrogada `ID_POBREZA`.

In [6]:
# Seleccionar y obtener combinaciones únicas
dim_pobreza = df[['POBREZA', 'POBREZAV', 'EN_DEFICIT']].drop_duplicates().reset_index(drop=True)

# Generar clave primaria
dim_pobreza.insert(0, 'ID_POBREZA', range(1, len(dim_pobreza) + 1))

print(f"Dimensión Pobreza: {dim_pobreza.shape[0]} registros únicos.")
display(dim_pobreza.head(3))

Dimensión Pobreza: 8 registros únicos.


,ID_POBREZA,POBREZA,POBREZAV,EN_DEFICIT
0,1,No Pobre,No Vulnerable,0
1,2,No Pobre,Vulnerable No Pobre,0
2,3,No Pobre,No Vulnerable,1


### 3.4. Tabla de Hechos (`fact_hogares`)
Construimos la tabla de hechos cruzando el dataset original con la `dim_pobreza` para recuperar el `ID_POBREZA`. 
Luego, conservamos únicamente las claves primarias/foráneas y las métricas (medidas continuas, logaritmos y proporciones).

In [7]:
# Mapear las dimensiones al dataframe principal
df_facts = df.merge(dim_geografia, on=['UBIGEO', 'DEPARTAMENTO', 'DOMINIO', 'ESTRATO'], how='left')
df_facts = df_facts.merge(dim_pobreza, on=['POBREZA', 'POBREZAV', 'EN_DEFICIT'], how='left')

# Seleccionar claves (PK, FK) y métricas
claves = ['ID_HOGAR', 'ID_GEOGRAFIA', 'MES_NUM', 'ID_POBREZA']
metricas = [
    'MIEPERHO', 'FACTOR07', 'INGHOG2D', 'GASHOG2D', 'BRECHA_HOG',
    'ING_PERCAPITA', 'GAS_PERCAPITA', 'BRECHA_PERCAPITA',
    'LOG_INGHOG2D', 'LOG_GASHOG2D', 'LOG_BRECHA_PERCAP'
]
# Seleccionar dinámicamente los grupos de gasto y sus porcentajes
columnas_grupos = [col for col in df.columns if col.startswith('GRU')]

columnas_finales = claves + metricas + columnas_grupos
fact_hogares = df_facts[columnas_finales]

print(f"Tabla de Hechos: {fact_hogares.shape[0]} filas, {fact_hogares.shape[1]} columnas.")
display(fact_hogares.head(3))

Tabla de Hechos: 33691 filas, 31 columnas.


,ID_HOGAR,ID_GEOGRAFIA,MES_NUM,ID_POBREZA,MIEPERHO,FACTOR07,INGHOG2D,GASHOG2D,BRECHA_HOG,ING_PERCAPITA,GAS_PERCAPITA,BRECHA_PERCAPITA,LOG_INGHOG2D,LOG_GASHOG2D,LOG_BRECHA_PERCAP,GRU11HD,GRU21HD,GRU31HD,GRU41HD,GRU51HD,GRU61HD,GRU71HD,GRU81HD,GRU11HD_PCT,GRU21HD_PCT,GRU31HD_PCT,GRU41HD_PCT,GRU51HD_PCT,GRU61HD_PCT,GRU71HD_PCT,GRU81HD_PCT
0,1,1,1,1,2,79.816757,52162.609375,34188.218750,17974.390625,26081.304688,17094.109375,8987.195312,10.862140,10.439666,9.103667,3845.363770,874.391602,1006.782715,1307.083496,3339.000000,3247.260742,1318.787598,3219.265869,0.1125,0.0256,0.0294,0.0382,0.0977,0.0950,0.0386,0.0942
1,2,1,1,1,3,79.816757,40832.042969,40164.945312,667.097656,13610.680990,13388.315104,222.365885,10.617247,10.600775,5.408811,12792.489258,2384.777832,916.000000,665.926880,2408.242676,2768.000000,895.084717,752.953064,0.3185,0.0594,0.0228,0.0166,0.0600,0.0689,0.0223,0.0187
2,3,1,1,1,1,79.816757,15098.497070,12308.838867,2789.658203,15098.497070,12308.838867,2789.658203,9.622417,9.418154,7.934033,2150.485107,603.583740,2589.000000,581.172546,2047.000000,233.000000,41.478962,557.595886,0.1747,0.0490,0.2103,0.0472,0.1663,0.0189,0.0034,0.0453


## 4. Validación de Integridad
Antes de exportar, verificaremos que no haya pérdida de datos reconstruyendo un fragmento de la tabla original a partir del Esquema Estrella.

In [8]:
# Hacemos join de la tabla de hechos con las dimensiones
df_reconstruido = fact_hogares.merge(dim_geografia, on='ID_GEOGRAFIA', how='inner') \
                              .merge(dim_tiempo, on='MES_NUM', how='inner') \
                              .merge(dim_pobreza, on='ID_POBREZA', how='inner')

# Verificamos la cantidad de filas
assert fact_hogares.shape[0] == df.shape[0], "Error: Pérdida de filas en la tabla de hechos."
assert df_reconstruido.shape[0] == df.shape[0], "Error: Pérdida de filas al reconstruir el esquema."
print("¡Integridad referencial validada! 0 registros perdidos.")

¡Integridad referencial validada! 0 registros perdidos.


## 5. Exportación de Modelos Básicos
Exportamos las cuatro tablas del modelo estrella para su consumo en Tableau.

In [9]:
# Crear directorio para el esquema estrella
os.makedirs('../Data/modelo/esquema_estrella', exist_ok=True)

# Exportar Dimensiones y Hechos
dim_geografia.to_csv('../Data/modelo/esquema_estrella/dim_geografia.csv', index=False, encoding='utf-8-sig')
dim_tiempo.to_csv('../Data/modelo/esquema_estrella/dim_tiempo.csv', index=False, encoding='utf-8-sig')
dim_pobreza.to_csv('../Data/modelo/esquema_estrella/dim_pobreza.csv', index=False, encoding='utf-8-sig')
fact_hogares.to_csv('../Data/modelo/esquema_estrella/fact_hogares.csv', index=False, encoding='utf-8-sig')

print("¡Modelo de Esquema Estrella exportado con éxito!")

¡Modelo de Esquema Estrella exportado con éxito!


## 6. Modelo 3: Esquema Copo de Nieve (Snowflake Schema)
Para tener una tercera opción y evaluar la máxima compresión, normalizamos la `dim_geografia` hasta la tercera forma normal (3NF), separando el departamento en su propia tabla.

In [10]:
# Rompemos dim_geografia en dos
dim_departamento = dim_geografia[['DEPARTAMENTO']].drop_duplicates().reset_index(drop=True)
dim_departamento.insert(0, 'ID_DEP', range(1, len(dim_departamento) + 1))

dim_geografia_snow = dim_geografia.merge(dim_departamento, on='DEPARTAMENTO', how='left')
dim_geografia_snow = dim_geografia_snow[['ID_GEOGRAFIA', 'UBIGEO', 'ID_DEP', 'DOMINIO', 'ESTRATO']]

os.makedirs('../Data/modelo/copo_de_nieve', exist_ok=True)
dim_departamento.to_csv('../Data/modelo/copo_de_nieve/dim_departamento.csv', index=False, encoding='utf-8-sig')
dim_geografia_snow.to_csv('../Data/modelo/copo_de_nieve/dim_geografia_snow.csv', index=False, encoding='utf-8-sig')
dim_tiempo.to_csv('../Data/modelo/copo_de_nieve/dim_tiempo.csv', index=False, encoding='utf-8-sig')
dim_pobreza.to_csv('../Data/modelo/copo_de_nieve/dim_pobreza.csv', index=False, encoding='utf-8-sig')
fact_hogares.to_csv('../Data/modelo/copo_de_nieve/fact_hogares.csv', index=False, encoding='utf-8-sig')
print("¡Esquema Copo de Nieve exportado!")

¡Esquema Copo de Nieve exportado!


## 7. Benchmarking Analítico (Prueba de Evidencia)
Calculamos el tamaño real en memoria (RAM) que consume cada arquitectura para elegir la mejor opción basándonos en datos empíricos de redundancia (Sparsity).

In [11]:
def calcular_memoria(dfs):
    total_bytes = sum(d.memory_usage(deep=True).sum() for d in dfs)
    return total_bytes / (1024 * 1024) # A Megabytes

mem_flat = calcular_memoria([df])
mem_star = calcular_memoria([fact_hogares, dim_geografia, dim_tiempo, dim_pobreza])
mem_snow = calcular_memoria([fact_hogares, dim_geografia_snow, dim_departamento, dim_tiempo, dim_pobreza])

resultados = pd.DataFrame({
    'Modelo': ['Tabla Plana (Base)', 'Esquema Estrella (Opcion 1)', 'Copo de Nieve (Opcion 2)'],
    'Memoria RAM (MB)': [mem_flat, mem_star, mem_snow],
    'Reduccion vs Base (%)': [0, (1 - mem_star/mem_flat)*100, (1 - mem_snow/mem_flat)*100],
    'Saltos de Join Topologicos': [0, 1, 2]
})

# Simulación a 5 años (Escalabilidad Histórica)
df_5y = pd.concat([df]*5, ignore_index=True)
fact_hogares_5y = pd.concat([fact_hogares]*5, ignore_index=True)

mem_flat_5y = calcular_memoria([df_5y])
mem_star_5y = calcular_memoria([fact_hogares_5y, dim_geografia, dim_tiempo, dim_pobreza])
mem_snow_5y = calcular_memoria([fact_hogares_5y, dim_geografia_snow, dim_departamento, dim_tiempo, dim_pobreza])

resultados['Memoria Proyectada 5 Años (MB)'] = [mem_flat_5y, mem_star_5y, mem_snow_5y]

display(resultados.round(2))
resultados.to_csv('../Data/modelo/benchmarking_resultados.csv', index=False, encoding='utf-8-sig')

,Modelo,Memoria RAM (MB),Reduccion vs Base (%),Saltos de Join Topologicos,Memoria Proyectada 5 Años (MB)
0,Tabla Plana (Base),12.15,0.00,0,60.73
1,Esquema Estrella (Opcion 1),8.15,32.88,1,40.03
2,Copo de Nieve (Opcion 2),8.14,32.98,2,40.01
